In [1]:
import os, json, time
import numpy as np, pandas as pd
import mlflow, mlflow.sklearn
from dotenv import load_dotenv
from mlflow.tracking import MlflowClient
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, recall_score, classification_report
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

load_dotenv("../.env")
mlflow.set_experiment("ids-multiclasse")

2026/11/20 00:54:45 INFO mlflow.tracking.fluent: Experiment with name 'ids-multiclasse' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/dc0676fe29fb4150aa6a0f39a7a055ad', creation_time=1795132480137, experiment_id='1', last_update_time=1795132480137, lifecycle_stage='active', name='ids-multiclasse', tags={}>

In [2]:
df = pd.read_parquet("../data/processed/flows_clean.parquet")

# regrouper les classes trop rares (moins de 100 lignes) dans "Rare_attack"
comptes = df["Label"].value_counts()
rares = comptes[comptes < 100].index
print("classes regroupées :", list(rares))
label = df["Label"].where(~df["Label"].isin(rares), "Rare_attack")

# transformer les noms en numéros (0, 1, 2...) : les modèles ne lisent que des nombres
le = LabelEncoder()
y = pd.Series(le.fit_transform(label), index=df.index)
print(dict(enumerate(le.classes_)))

# colonnes d'entrée (les mêmes que pour le binaire)
FEATURES = [c for c in df.columns if c not in ("Label", "is_attack")]
X = df[FEATURES].astype("float32")

# sauvegarder la liste des noms de classes (le streaming en aura besoin)
os.makedirs("../models", exist_ok=True)
with open("../models/label_classes.json", "w") as f:
    json.dump(list(le.classes_), f)

classes regroupées : ['Infiltration', 'Web Attack - Sql Injection', 'Heartbleed']
{0: 'Benign', 1: 'Bot', 2: 'DDoS', 3: 'DoS GoldenEye', 4: 'DoS Hulk', 5: 'DoS Slowhttptest', 6: 'DoS slowloris', 7: 'FTP-Patator', 8: 'PortScan', 9: 'Rare_attack', 10: 'SSH-Patator', 11: 'Web Attack - Brute Force', 12: 'Web Attack - XSS'}


In [3]:
X_tmp, X_test, y_tmp, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(
    X_tmp, y_tmp, test_size=0.1765, stratify=y_tmp, random_state=42)
print(X_train.shape, X_val.shape, X_test.shape)

(1539394, 77) (329937, 77) (329882, 77)


In [4]:
def run(nom, model, params, X_tr, y_tr, fit_kwargs=None):
    with mlflow.start_run(run_name=nom) as r:
        mlflow.log_params({"model": nom, "n_train_rows": len(X_tr), **params})

        t = time.time()
        model.fit(X_tr, y_tr, **(fit_kwargs or {}))
        duree = time.time() - t

        pred = model.predict(X_val)
        metrics = {
            "f1_macro": f1_score(y_val, pred, average="macro"),
            "f1_weighted": f1_score(y_val, pred, average="weighted"),
            "recall_macro": recall_score(y_val, pred, average="macro"),
            "fit_seconds": duree,
        }
        mlflow.log_metrics(metrics)

        # rapport détaillé : précision / recall / F1 pour chaque type d'attaque
        rapport = classification_report(y_val, pred, labels=range(len(le.classes_)),
                                        target_names=le.classes_, zero_division=0)
        mlflow.log_text(rapport, "classification_report.txt")

        mlflow.log_artifact("../models/label_classes.json")
        mlflow.sklearn.log_model(model, artifact_path="model")
        print(nom, {k: round(v, 4) for k, v in metrics.items()})
        return r.info.run_id

In [5]:
# 1) LightGBM
run("lightgbm_multi",
    LGBMClassifier(n_estimators=150, learning_rate=0.1,
                   class_weight="balanced", n_jobs=-1, random_state=42, verbose=-1),
    {"n_estimators": 150, "learning_rate": 0.1, "class_weight": "balanced"},
    X_train, y_train)

2026/11/20 01:01:42 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


lightgbm_multi {'f1_macro': 0.1814, 'f1_weighted': 0.6897, 'recall_macro': 0.4228, 'fit_seconds': 178.2623}
🏃 View run lightgbm_multi at: https://dagshub.com/hanamezghani/mlops-network-intrusion-detection.mlflow/#/experiments/1/runs/77f52774664b4c12a2b4a214c56e89c2
🧪 View experiment at: https://dagshub.com/hanamezghani/mlops-network-intrusion-detection.mlflow/#/experiments/1


'77f52774664b4c12a2b4a214c56e89c2'

In [6]:
# 2) XGBoost : pas de class_weight, on passe un poids par ligne (même effet)
w = compute_sample_weight("balanced", y_train)
run("xgboost_multi",
    XGBClassifier(n_estimators=150, learning_rate=0.1, tree_method="hist",
                  n_jobs=-1, random_state=42),
    {"n_estimators": 150, "learning_rate": 0.1, "sample_weight": "balanced"},
    X_train, y_train, fit_kwargs={"sample_weight": w})

2026/11/20 01:14:15 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


xgboost_multi {'f1_macro': 0.8739, 'f1_weighted': 0.9981, 'recall_macro': 0.9205, 'fit_seconds': 580.2905}
🏃 View run xgboost_multi at: https://dagshub.com/hanamezghani/mlops-network-intrusion-detection.mlflow/#/experiments/1/runs/b1dbf9219fd748239ba81a59ba328d37
🧪 View experiment at: https://dagshub.com/hanamezghani/mlops-network-intrusion-detection.mlflow/#/experiments/1


'b1dbf9219fd748239ba81a59ba328d37'

In [7]:
# 3) RandomForest : échantillon de 400 000 lignes pour rester rapide
X_rf, _, y_rf, _ = train_test_split(
    X_train, y_train, train_size=400_000, stratify=y_train, random_state=42)
run("random_forest_multi",
    RandomForestClassifier(n_estimators=100, max_depth=20, min_samples_leaf=5,
                           class_weight="balanced", n_jobs=-1, random_state=42),
    {"n_estimators": 100, "max_depth": 20, "min_samples_leaf": 5,
     "class_weight": "balanced"},
    X_rf, y_rf)

2026/11/20 01:15:32 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


random_forest_multi {'f1_macro': 0.7229, 'f1_weighted': 0.9918, 'recall_macro': 0.8696, 'fit_seconds': 14.4328}
🏃 View run random_forest_multi at: https://dagshub.com/hanamezghani/mlops-network-intrusion-detection.mlflow/#/experiments/1/runs/cda9a405e0d442b5a1dff30bd9da51da
🧪 View experiment at: https://dagshub.com/hanamezghani/mlops-network-intrusion-detection.mlflow/#/experiments/1


'cda9a405e0d442b5a1dff30bd9da51da'

In [8]:
# 1) trouver le meilleur essai selon f1_macro
runs = mlflow.search_runs(experiment_names=["ids-multiclasse"],
                          order_by=["metrics.f1_macro DESC"], max_results=5)
print(runs[["tags.mlflow.runName", "metrics.f1_macro", "run_id"]])
RUN_ID = runs.loc[0, "run_id"]

# 2) test final, une seule fois
best = mlflow.sklearn.load_model(f"runs:/{RUN_ID}/model")
pred = best.predict(X_test)
test_metrics = {
    "test_f1_macro": f1_score(y_test, pred, average="macro"),
    "test_f1_weighted": f1_score(y_test, pred, average="weighted"),
}
print(test_metrics)
with mlflow.start_run(run_id=RUN_ID):
    mlflow.log_metrics(test_metrics)

# 3) enregistrer dans le Registry avec l'alias "champion"
mv = mlflow.register_model(f"runs:/{RUN_ID}/model", "ids-multiclasse")
MlflowClient().set_registered_model_alias("ids-multiclasse", "champion", mv.version)
print("version", mv.version, "→ champion")

   tags.mlflow.runName  metrics.f1_macro                            run_id
0        xgboost_multi          0.873944  b1dbf9219fd748239ba81a59ba328d37
1  random_forest_multi          0.722871  cda9a405e0d442b5a1dff30bd9da51da
2       lightgbm_multi          0.181404  77f52774664b4c12a2b4a214c56e89c2


c:\Users\User\Documents\mlops-network-intrusion-detection\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'test_f1_macro': 0.8841113153363875, 'test_f1_weighted': 0.9980437872588402}
🏃 View run xgboost_multi at: https://dagshub.com/hanamezghani/mlops-network-intrusion-detection.mlflow/#/experiments/1/runs/b1dbf9219fd748239ba81a59ba328d37
🧪 View experiment at: https://dagshub.com/hanamezghani/mlops-network-intrusion-detection.mlflow/#/experiments/1


Successfully registered model 'ids-multiclasse'.
2026/11/20 01:16:49 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: ids-multiclasse, version 1
Created version '1' of model 'ids-multiclasse'.


version 1 → champion
